# Unveiling the Android App Market

**Oasis Infobyte Data Analytics Internship — Level 2, Task 4**

This notebook analyses the Google Play Store ecosystem using app metadata and user reviews.


## 1. Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load the Two Datasets Separately


In [ ]:
APPS_PATH = "googleplaystore.csv"
REVIEWS_PATH = "googleplaystore_user_reviews.csv"

apps_raw = pd.read_csv(APPS_PATH)
reviews_raw = pd.read_csv(REVIEWS_PATH)

print("Apps dataset:", apps_raw.shape)
print("Reviews dataset:", reviews_raw.shape)

display(apps_raw.head())
display(reviews_raw.head())


## 3. Initial Data Inspection


In [ ]:
print("Apps columns:")
display(apps_raw.dtypes.to_frame("dtype"))

print("\nApps missing values:")
display(
    apps_raw.isna().sum()
    .sort_values(ascending=False)
    .head(20)
    .to_frame("missing_values")
)

print("\nReview columns:")
display(reviews_raw.dtypes.to_frame("dtype"))

print("\nReview missing values:")
display(
    reviews_raw.isna().sum()
    .sort_values(ascending=False)
    .to_frame("missing_values")
)


## 4. Clean the Apps Dataset


In [ ]:
apps = apps_raw.copy()

# Remove exact duplicate rows first.
duplicate_rows = apps.duplicated().sum()
apps = apps.drop_duplicates().copy()

# Convert numeric-like fields.
apps["Reviews"] = pd.to_numeric(apps["Reviews"], errors="coerce")

apps["Rating"] = pd.to_numeric(apps["Rating"], errors="coerce")

apps["Installs"] = (
    apps["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)
apps["Installs"] = pd.to_numeric(apps["Installs"], errors="coerce")

apps["Price"] = (
    apps["Price"]
    .astype(str)
    .str.replace("$", "", regex=False)
)
apps["Price"] = pd.to_numeric(apps["Price"], errors="coerce")

# Convert Size to approximate MB.
def size_to_mb(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value.endswith("M"):
        return float(value[:-1])

    if value.endswith("k"):
        return float(value[:-1]) / 1024

    return np.nan

apps["Size_MB"] = apps["Size"].apply(size_to_mb)

# Clean Type and category labels.
apps["Type"] = apps["Type"].astype("string").str.strip().str.title()
apps["Category"] = apps["Category"].astype("string").str.strip()

print("Exact duplicate rows removed:", duplicate_rows)
print("Cleaned apps shape:", apps.shape)


## 5. Handle Missing Values


In [ ]:
missing_before = apps.isna().sum()

# Rating is only used for rating-specific analysis, so rows without ratings
# are retained in the general app dataset but excluded from rating calculations.

# Required fields for installs/pricing analysis.
apps = apps.dropna(subset=["App", "Category", "Installs", "Type", "Price"]).copy()

missing_after = apps.isna().sum()

missing_comparison = pd.DataFrame({
    "Before": missing_before,
    "After": missing_after
}).sort_values("Before", ascending=False)

display(missing_comparison.head(20))


## 6. Clean the User Reviews Dataset


In [ ]:
reviews = reviews_raw.copy()

review_duplicates = reviews.duplicated().sum()
reviews = reviews.drop_duplicates().copy()

reviews["App"] = reviews["App"].astype("string").str.strip()
reviews["Translated_Review"] = reviews["Translated_Review"].astype("string")

reviews = reviews.dropna(
    subset=["App", "Translated_Review"]
).copy()

print("Review duplicate rows removed:", review_duplicates)
print("Cleaned reviews shape:", reviews.shape)


## 7. Category Distribution and Saturation


In [ ]:
category_counts = (
    apps["Category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="App_Count")
)

display(category_counts.head(15))

plt.figure(figsize=(12, 7))
sns.barplot(
    data=category_counts,
    x="App_Count",
    y="Category"
)
plt.title("Number of Apps by Category")
plt.xlabel("Number of Apps")
plt.ylabel("Category")
plt.tight_layout()
plt.show()


### Saturation Interpretation

In this analysis, category saturation is approximated by the number of apps listed in each category.

A category with a larger app count is more saturated in terms of the number of competing listings. This is a market-structure signal, not a direct measurement of profitability, quality or competitive difficulty.


## 8. Ratings Analysis


In [ ]:
rating_data = apps.dropna(subset=["Rating"]).copy()

plt.figure(figsize=(9, 5))
sns.histplot(rating_data["Rating"], bins=20, kde=True)
plt.title("Distribution of App Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Apps")
plt.tight_layout()
plt.show()

avg_rating_category = (
    rating_data.groupby("Category")["Rating"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

display(avg_rating_category.head(15))


## 9. Size vs Installs


In [ ]:
size_installs = apps.dropna(
    subset=["Size_MB", "Installs"]
).copy()

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=size_installs.sample(
        min(5000, len(size_installs)),
        random_state=42
    ),
    x="Size_MB",
    y="Installs",
    alpha=0.5
)
plt.xscale("log")
plt.yscale("log")
plt.title("App Size vs Number of Installs")
plt.xlabel("App Size (MB, log scale)")
plt.ylabel("Installs (log scale)")
plt.tight_layout()
plt.show()

size_install_correlation = size_installs["Size_MB"].corr(
    size_installs["Installs"]
)

print(f"Pearson correlation between size and installs: {size_install_correlation:.4f}")


### Correlation Note

A correlation value describes linear association between the available numeric observations. It does not establish that app size causes higher or lower installs.

The strong skew in installs also means a log-scale visualisation is useful for interpreting the distribution.


## 10. Free vs Paid Apps


In [ ]:
type_counts = apps["Type"].value_counts()

display(type_counts.to_frame("App_Count"))

plt.figure(figsize=(7, 5))
sns.countplot(data=apps, x="Type")
plt.title("Free vs Paid Apps")
plt.xlabel("App Type")
plt.ylabel("Number of Apps")
plt.tight_layout()
plt.show()


## 11. Paid App Price Distribution


In [ ]:
paid_apps = apps[
    (apps["Type"].str.lower() == "Paid") &
    (apps["Price"] > 0)
].copy()

plt.figure(figsize=(10, 5))
sns.histplot(paid_apps["Price"], bins=40)
plt.xlim(0, paid_apps["Price"].quantile(0.99))
plt.title("Paid App Price Distribution")
plt.xlabel("Price")
plt.ylabel("Number of Apps")
plt.tight_layout()
plt.show()

display(
    paid_apps["Price"]
    .describe()
    .to_frame("Price")
)


## 12. Revenue Estimate by Category


In [ ]:
# Simple gross-revenue proxy:
# estimated_revenue = listed_price × install_count

paid_apps["Estimated_Revenue"] = (
    paid_apps["Price"] * paid_apps["Installs"]
)

revenue_by_category = (
    paid_apps.groupby("Category")["Estimated_Revenue"]
    .agg(["sum", "count"])
    .sort_values("sum", ascending=False)
    .rename(columns={
        "sum": "Estimated_Gross_Revenue",
        "count": "Paid_App_Count"
    })
)

display(revenue_by_category.head(15))

plt.figure(figsize=(12, 7))
sns.barplot(
    data=revenue_by_category.head(15).reset_index(),
    x="Estimated_Gross_Revenue",
    y="Category"
)
plt.title("Estimated Gross Revenue by Category — Top 15")
plt.xlabel("Estimated Gross Revenue")
plt.ylabel("Category")
plt.tight_layout()
plt.show()


### Revenue Caveat

The revenue calculation is a simple proxy:

**Price × Installs**

It is not actual developer revenue. It ignores Google Play fees, taxes, refunds, discounts, regional pricing, subscriptions and in-app purchases.

Therefore, use it for relative exploratory comparison rather than financial reporting.


## 13. VADER Sentiment Analysis


In [ ]:
analyzer = SentimentIntensityAnalyzer()

def vader_label(text):
    score = analyzer.polarity_scores(str(text))["compound"]

    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    return "Neutral"

reviews["VADER_Sentiment"] = reviews["Translated_Review"].apply(vader_label)

sentiment_counts = reviews["VADER_Sentiment"].value_counts()

display(sentiment_counts.to_frame("Review_Count"))

plt.figure(figsize=(8, 5))
sns.countplot(
    data=reviews,
    x="VADER_Sentiment",
    order=["Positive", "Neutral", "Negative"]
)
plt.title("User Review Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()


## 14. Join Reviews with App Categories


In [ ]:
app_category_lookup = (
    apps[["App", "Category"]]
    .drop_duplicates(subset=["App"])
)

reviews_with_category = reviews.merge(
    app_category_lookup,
    on="App",
    how="left"
)

category_sentiment = (
    reviews_with_category
    .dropna(subset=["Category"])
    .groupby(["Category", "VADER_Sentiment"])
    .size()
    .unstack(fill_value=0)
)

for sentiment in ["Positive", "Neutral", "Negative"]:
    if sentiment not in category_sentiment.columns:
        category_sentiment[sentiment] = 0

category_sentiment["Total"] = category_sentiment[
    ["Positive", "Neutral", "Negative"]
].sum(axis=1)

category_sentiment["Positive_Rate"] = (
    category_sentiment["Positive"] / category_sentiment["Total"]
)

category_sentiment["Negative_Rate"] = (
    category_sentiment["Negative"] / category_sentiment["Total"]
)

display(
    category_sentiment
    .sort_values("Positive_Rate", ascending=False)
    .head(15)
)


## 15. Categories with Most Positive and Negative Sentiment


In [ ]:
positive_categories = (
    category_sentiment
    .sort_values("Positive_Rate", ascending=False)
    .head(10)
)

negative_categories = (
    category_sentiment
    .sort_values("Negative_Rate", ascending=False)
    .head(10)
)

print("Highest positive sentiment rates:")
display(
    positive_categories[
        ["Positive", "Neutral", "Negative", "Total", "Positive_Rate"]
    ]
)

print("Highest negative sentiment rates:")
display(
    negative_categories[
        ["Positive", "Neutral", "Negative", "Total", "Negative_Rate"]
    ]
)


## 16. Interactive Plotly Visualisation


In [ ]:
interactive_category = (
    apps.groupby("Category")
    .agg(
        App_Count=("App", "count"),
        Average_Rating=("Rating", "mean"),
        Total_Installs=("Installs", "sum")
    )
    .reset_index()
)

fig = px.scatter(
    interactive_category,
    x="App_Count",
    y="Average_Rating",
    size="Total_Installs",
    hover_name="Category",
    title="Interactive Category Overview",
    labels={
        "App_Count": "Number of Apps",
        "Average_Rating": "Average Rating",
        "Total_Installs": "Total Installs"
    }
)

fig.show()


## 17. Three Data-Driven Developer Insights


In [ ]:
# Generate supporting rankings for the conclusion.
most_saturated = category_counts.iloc[0]["Category"]

best_rated_category = (
    avg_rating_category[avg_rating_category["count"] >= 5]
    .sort_values("mean", ascending=False)
    .index[0]
    if len(avg_rating_category[avg_rating_category["count"] >= 5]) else "Not enough data"
)

top_revenue_category = (
    revenue_by_category.index[0]
    if len(revenue_by_category) else "No paid-app revenue data"
)

print("Insight 1 — Market saturation:")
print(
    f"{most_saturated} has the largest number of app listings in the cleaned dataset, "
    "which indicates a relatively crowded category by app count."
)

print("\nInsight 2 — Ratings:")
print(
    f"{best_rated_category} has the highest average rating among categories "
    "meeting the minimum observation threshold used above."
)

print("\nInsight 3 — Monetisation:")
print(
    f"{top_revenue_category} has the highest estimated gross revenue under the "
    "Price × Installs proxy used in this analysis."
)

print("\nDeveloper implication:")
print(
    "A developer should balance category crowding, user sentiment/ratings, "
    "and monetisation potential rather than choosing a category using a single metric."
)


## 18. Conclusion


In [ ]:
summary = {
    "Most saturated category by app count": most_saturated,
    "Highest-rated category under threshold": best_rated_category,
    "Top estimated revenue category": top_revenue_category,
    "Size-install correlation": round(size_install_correlation, 4)
}

display(pd.DataFrame(summary.items(), columns=["Metric", "Result"]))

print(
    "The analysis combines ecosystem structure, ratings, installs, pricing and "
    "user sentiment to provide a data-driven view of the Google Play Store."
)


## 19. Save Key Outputs


In [ ]:
category_counts.to_csv(
    "category_distribution.csv",
    index=False
)

avg_rating_category.to_csv(
    "average_rating_by_category.csv"
)

revenue_by_category.to_csv(
    "estimated_revenue_by_category.csv"
)

category_sentiment.to_csv(
    "sentiment_by_category.csv"
)

print("Key analysis outputs saved successfully.")
